[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Tables and Metadata &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/college.db` as the notebook's Setup did, and makes what its worked
examples made: `show_ddl`, the naming convention `NAMING`, the `college` schema and `build_college`,
a database in memory called `memory`, the Setup database's engine `disk`, and the registrar's
database, built from `college`. Run it first. The tasks do not depend on one another, and the last
cell removes the scratch folder.


In [1]:
import logging
import shutil
import sqlite3
from datetime import date
from pathlib import Path

import sqlalchemy
from sqlalchemy import (CheckConstraint, Column, Date, ForeignKey, Integer, MetaData, String, Table, UniqueConstraint,
                        create_engine, event, func, insert, inspect, select, text)
from sqlalchemy.exc import IntegrityError, OperationalError
from sqlalchemy.pool import StaticPool
from sqlalchemy.schema import CreateTable

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT NOT NULL UNIQUE,
                           program TEXT NOT NULL, started_on TEXT NOT NULL);
    CREATE TABLE courses (id INTEGER PRIMARY KEY, code TEXT NOT NULL UNIQUE, title TEXT NOT NULL,
                          department TEXT NOT NULL, credits INTEGER NOT NULL);
    CREATE TABLE terms (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE, starts_on TEXT NOT NULL);
    CREATE TABLE sections (id INTEGER PRIMARY KEY, course_id INTEGER NOT NULL REFERENCES courses (id),
                           term_id INTEGER NOT NULL REFERENCES terms (id), capacity INTEGER NOT NULL);
    CREATE TABLE enrollments (student_id INTEGER NOT NULL REFERENCES students (id),
                              section_id INTEGER NOT NULL REFERENCES sections (id),
                              status TEXT NOT NULL, grade TEXT,
                              PRIMARY KEY (student_id, section_id));
""")
build.executemany("INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)", STUDENTS)
build.executemany("INSERT INTO courses (code, title, department, credits) VALUES (?, ?, ?, ?)", COURSES)
build.executemany("INSERT INTO terms (name, starts_on) VALUES (?, ?)", TERMS)
build.executemany("INSERT INTO sections (course_id, term_id, capacity) VALUES (?, ?, ?)", SECTIONS)
build.executemany("INSERT INTO enrollments (student_id, section_id, status, grade) VALUES (?, ?, ?, ?)", ENROLLMENTS)
build.commit()
build.close()

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine


def show_ddl(table, engine):
    """Print the CREATE TABLE statement a Table becomes on an engine's database."""
    for line in str(CreateTable(table).compile(engine)).strip().splitlines():
        print("   ", line.rstrip().replace("\t", "    "))


NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}
college = MetaData(naming_convention=NAMING)

students = Table(
    "students", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(100), nullable=False),
    Column("email", String(200), nullable=False, unique=True),
    Column("program", String(50), nullable=False),
    Column("started_on", Date, nullable=False),
)
courses = Table(
    "courses", college,
    Column("id", Integer, primary_key=True),
    Column("code", String(10), nullable=False, unique=True),
    Column("title", String(100), nullable=False),
    Column("department", String(50), nullable=False),
    Column("credits", Integer, nullable=False),
    CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),
)
terms = Table(
    "terms", college,
    Column("id", Integer, primary_key=True),
    Column("name", String(20), nullable=False, unique=True),
    Column("starts_on", Date, nullable=False),
)
sections = Table(
    "sections", college,
    Column("id", Integer, primary_key=True),
    Column("course_id", ForeignKey("courses.id"), nullable=False),
    Column("term_id", ForeignKey("terms.id"), nullable=False),
    Column("capacity", Integer, nullable=False),
    UniqueConstraint("course_id", "term_id"),
    CheckConstraint("capacity > 0", name="capacity_positive"),
)
enrollments = Table(
    "enrollments", college,
    Column("student_id", ForeignKey("students.id"), primary_key=True),
    Column("section_id", ForeignKey("sections.id"), primary_key=True),
    Column("status", String(20), nullable=False, server_default="enrolled"),
    Column("grade", String(2)),
    CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),
)


def build_college(engine):
    """Create the college's tables from `college`, load the lists from Setup into them, and count their rows."""
    college.create_all(engine)
    rows = {
        students: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                   for name, email, program, started in STUDENTS],
        courses: [{"code": code, "title": title, "department": department, "credits": credits}
                  for code, title, department, credits in COURSES],
        terms: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        sections: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        enrollments: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                      for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for table in college.sorted_tables:
            conn.execute(insert(table), rows[table])
        return {table.name: conn.execute(select(func.count()).select_from(table)).scalar_one()
                for table in college.sorted_tables}


memory = college_engine()
disk = college_engine(DATABASE)
registrar = college_engine(SCRATCH / "registrar.db")
build_college(registrar)
print("sqlalchemy", sqlalchemy.__version__, "|", inspect(registrar).get_table_names())


sqlalchemy 2.0.54 | ['courses', 'enrollments', 'sections', 'students', 'terms']


**1.** A `rooms` table, with a unique pair of columns.


In [2]:
campus = MetaData(naming_convention=NAMING)
rooms = Table(
    "rooms", campus,
    Column("id", Integer, primary_key=True),
    Column("building", String(50), nullable=False),
    Column("number", String(10), nullable=False),
    Column("seats", Integer),
    UniqueConstraint("building", "number"),
)
show_ddl(rooms, memory)


    CREATE TABLE rooms (
        id INTEGER NOT NULL,
        building VARCHAR(50) NOT NULL,
        number VARCHAR(10) NOT NULL,
        seats INTEGER,
        CONSTRAINT pk_rooms PRIMARY KEY (id),
        CONSTRAINT uq_rooms_building_number UNIQUE (building, number)
    )


`UniqueConstraint` with two columns makes the pair unique, so two rooms may share a building or a
number but not both, and the naming convention joined both column names into its name.


**2.** A named check, and the names of every constraint.


In [3]:
rooms.append_constraint(CheckConstraint("seats > 0", name="seats_positive"))

for constraint in sorted(rooms.constraints, key=lambda constraint: constraint.name):
    print(f"{type(constraint).__name__:<21}", constraint.name)


CheckConstraint       ck_rooms_seats_positive
PrimaryKeyConstraint  pk_rooms
UniqueConstraint      uq_rooms_building_number


`append_constraint` adds a constraint to a table that already exists as an object.
`rooms.constraints` is a set, whose order can change from one run to the next, so the loop sorts it
by name.


**3.** One table, reflected.


In [4]:
found_courses = Table("courses", MetaData(), autoload_with=disk)
for column in found_courses.columns:
    print(f"{column.name:<11}", repr(column.type))


id          INTEGER()
code        TEXT()
title       TEXT()
department  TEXT()
credits     INTEGER()


The types are the ones in Setup's SQL text: `INTEGER` for the id and the credits, and `TEXT` for the
rest.


**4.** The foreign keys of `sections`, from the inspector.


In [5]:
for key in sorted(inspect(disk).get_foreign_keys("sections"), key=lambda key: key["constrained_columns"]):
    print(key["constrained_columns"], "->", key["referred_table"], key["referred_columns"], "| name:", key["name"])


['course_id'] -> courses ['id'] | name: None
['term_id'] -> terms ['id'] | name: None


**5.** Two rooms, and a third that repeats one.


In [6]:
campus.create_all(memory)
with memory.begin() as conn:
    conn.execute(insert(rooms), [{"building": "Hall A", "number": "101", "seats": 40},
                                 {"building": "Hall A", "number": "102", "seats": 25}])
try:
    with memory.begin() as conn:
        conn.execute(insert(rooms).values(building="Hall A", number="101", seats=60))
except IntegrityError as error:
    print(str(error).splitlines()[0])


(sqlite3.IntegrityError) UNIQUE constraint failed: rooms.building, rooms.number


The error names both columns of the pair, since a unique constraint is reported by its columns, and
the block rolled back, so the room with 60 seats was never added.


**6.** Drop the college, and build it again.


In [7]:
college.drop_all(registrar)
print("after drop_all:     ", inspect(registrar).get_table_names())
print("after build_college:", build_college(registrar))


after drop_all:      []
after build_college: {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}


`drop_all` drops the tables in the reverse of `sorted_tables`, so no table disappears while another
still refers to it, and `build_college` creates and loads them again from the lists in Setup.

Last, remove the scratch folder:


In [8]:
for engine in (memory, disk, registrar):
    engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Tables and Metadata](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/05-tables-and-metadata.ipynb)  &nbsp;&middot;&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
